<a href="https://colab.research.google.com/github/arnabd64/Google-Colab-Notebooks/blob/main/TNER_wikineural_multi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Named Entity Recognition

In [ ]:
! pip install --progress-bar=off \
    transformers[torch] \
    datasets \
    evaluate \
    seqeval \
> install.log

## Log in to Huggingface Hub

In [ ]:
import huggingface_hub
from google.colab import userdata

huggingface_hub.login(userdata.get("HF"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


## Set Model & Dataset

In [ ]:
DATASET = "arnabdhar/wikiner-multilingual"
MODEL = "distilbert-base-cased-mult"
OUTPUT = "xlm-roberta-base-wikineural"

## Load Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

## Setup Labels

In [ ]:
LABEL2ID = {
"O": 0,
"B-PER": 1,
"I-PER": 2,
"B-LOC": 3,
"I-LOC": 4,
"B-ORG": 5,
"I-ORG": 6,
"B-ANIM": 7,
"I-ANIM": 8,
"B-BIO": 9,
"I-BIO": 10,
"B-CEL": 11,
"I-CEL": 12,
"B-DIS": 13,
"I-DIS": 14,
"B-EVE": 15,
"I-EVE": 16,
"B-FOOD": 17,
"I-FOOD": 18,
"B-INST": 19,
"I-INST": 20,
"B-MEDIA": 21,
"I-MEDIA": 22,
"B-PLANT": 23,
"I-PLANT": 24,
"B-MYTH": 25,
"I-MYTH": 26,
"B-TIME": 27,
"I-TIME": 28,
"B-VEHI": 29,
"I-VEHI": 30,
"B-SUPER": 31,
"I-SUPER": 32,
"B-PHY": 33,
"I-PHY": 34
}

LABELS = [key for key,value in LABEL2ID.items()]
ID2LABEL = {value:key for key, value in LABEL2ID.items()}

## Load Dataset

In [ ]:
import datasets
from datasets import Features, Sequence, Value, ClassLabel
wikiner = datasets.load_dataset("arnabdhar/wikiner-multilingual",
                                features = Features({
                                    "tokens": Sequence(Value("string")),
                                    "tags": Sequence(ClassLabel(num_classes=len(LABELS), names=LABELS))
                                }))

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/2506842 [00:00<?, ? examples/s]

## Preprocess the Dataset

In [ ]:
tokenize = lambda batch: tokenizer(batch, is_split_into_words=True, truncation=True, padding="max_length", max_length=64)

# def rename_labels(batch):
#     # renamed_labels = list()
#     # for labels in batch['tags']:
#     #     renamed_labels.append([ID2LABEL[l] for l in labels])
#     # batch['tags'] = renamed_labels
#     batch['tags'] = [[ID2LABEL[l] for l in labels] for labels in batch['tags']]
#     return batch


def tokenize_and_align(batch):
    tokenized_inputs = tokenize(batch['tokens'])
    aligned_labels_batch = list()
    for i, label in enumerate(batch['tags']):
        aligned_labels_sentence = list()
        for word_id in tokenized_inputs.word_ids(batch_index = i):
            if word_id is None:
                aligned_labels_sentence.append(-100)
            else:
                aligned_labels_sentence.append(label[word_id])
        aligned_labels_batch.append(aligned_labels_sentence)
    tokenized_inputs['labels'] = aligned_labels_batch
    return tokenized_inputs


# wiki = wiki.map(rename_labels, batched=True, batch_size=1000)
# wiki = wiki.cast_column("tags", Sequence(ClassLabel(num_classes=len(LABELS), names=LABELS)))
wiki = wikiner.map(tokenize_and_align, batched=True)
wiki = wiki.remove_columns(['tokens','tags'])
wiki = wikiner.train_test_split(test_size=0.1, shuffle=True)

Map:   0%|          | 0/2506842 [00:00<?, ? examples/s]

ArrowInvalid: ignored

In [ ]:
wiki

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 924755
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 102751
    })
})

## Evaluation Function

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    # extract output from model
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = list()
    true_labels = list()

    for prediction, label in zip(predictions, labels):
        batch_predictions = list()
        batch_labels = list()
        for p, l in zip(prediction, label):
            if l != -100:
                batch_predictions.append(LABELS[p])
                batch_labels.append(LABELS[l])
        true_predictions.append(batch_predictions)
        true_labels.append(batch_labels)

    # compute metrics
    metrics = seqeval.compute(predictions=true_predictions, references=true_labels)
    return dict(
        recall = metrics['overall_recall'],
        precision = metrics['overall_precision'],
        f1 = metrics['overall_f1'],
        accuracy = metrics['overall_accuracy']
    )

## Setup Training

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForTokenClassification, DataCollatorForTokenClassification

args = TrainingArguments(
    # output model
    output_dir = OUTPUT,
    save_safetensors = True,
    save_total_limit = 5,
    load_best_model_at_end=True,
    metric_for_best_model = "eval_loss",
    push_to_hub = True,

    # Training & Evaluation length
    max_steps = 90_000,
    eval_steps = 10_000,
    save_steps = 30_000,

    # Batch
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 128,
    dataloader_num_workers = 2,

    # Optimizer
    optim = "adamw_torch",
    learning_rate=4e-5,
    weight_decay=1e-9,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Others
    evaluation_strategy='steps',
    save_strategy='steps',
    report_to=['tensorboard'],
    seed=37912547
)

trainer = Trainer(
    model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID),
    tokenizer = tokenizer,
    args = args,
    compute_metrics = compute_metrics,
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding="max_length", max_length=64),
    train_dataset = wiki['train'],
    eval_dataset = wiki['test']
)

output = trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10000,0.085800,0.081692
20000,0.071900,0.066042
30000,0.065600,0.063123
40000,0.059800,0.057387
50000,0.055100,0.053394
60000,0.052300,0.051246
70000,0.051900,0.048378
80000,0.041800,0.048042
90000,0.042000,0.046895
100000,0.041000,0.046697


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with 

## Metrics

In [1]:
trainer.log_metrics("validation", trainer.evaluate(wikiner['test']))

NameError: name 'trainer' is not defined

## Push model to Hub

In [ ]:
trainer.push_to_hub(tags=['pytorch','XLMRobertaForTokenClassification','named-entity-recognition','wikipedia'])

events.out.tfevents.1702165372.584b4064680c.508.4:   0%|          | 0.00/364 [00:00<?, ?B/s]

'https://huggingface.co/arnabdhar/xlm-roberta-base-wikineural/tree/main/'